In [1]:
# Run everytime a new function is added
import numpy as np
from scripts.utilities import *
from scripts.features import *

In [2]:
consumer_df, account_df, transaction_df = get_data()
transaction_df.amount = transaction_df.amount.apply(abs)

Data successfully loaded and processed.


In [3]:
c_df = consumer_df.dropna(subset='DQ_TARGET')
a_df = account_df[account_df.prism_consumer_id.isin(c_df.prism_consumer_id)]
t_df = transaction_df[transaction_df.prism_consumer_id.isin(c_df.prism_consumer_id)]

result = t_df[['prism_consumer_id']].drop_duplicates().reset_index(drop=True)
# display(c_df, a_df, t_df)

## Account balance overtime:

- Balance recorded at the time account_df was made:

In [4]:
balance = a_df.groupby(['prism_consumer_id']).agg({'balance_date':'max', 'balance':'sum'})
display(balance)
acct_balance = balance['balance']
acct_balance

,balance_date,balance
prism_consumer_id,,
0,2021-08-31,320.37
1,2021-06-30,3302.42
2,2021-04-30,2805.36
3,2021-02-28,7667.01
4,2021-09-30,394.55
...,...,...
13995,2022-01-22,1028.80
13996,2022-02-01,11495.77
13997,2021-12-15,2396.85


prism_consumer_id
0          320.37
1         3302.42
2         2805.36
3         7667.01
4          394.55
           ...   
13995     1028.80
13996    11495.77
13997     2396.85
13998    14835.71
13999      -41.00
Name: balance, Length: 10408, dtype: float64

In [5]:
# t_df['amount'] = np.where(t_df['credit_or_debit'] == 'DEBIT', -t_df['amount'], t_df['amount'])
# t_df

- Current balance:

In [6]:
t = t_df.copy()

In [7]:
t['balance_date'] = balance['balance_date']
t['amount'] = np.where(t['credit_or_debit'] == 'DEBIT', -t['amount'], t['amount'])
t['is_before_balance_date'] = np.where(t['posted_date'] < t['balance_date'], True, False)
update_balance = t[t['is_before_balance_date'] == False]
update_balance = update_balance.groupby(['prism_consumer_id'])['amount'].sum()

In [8]:
current_balance = c_df[['prism_consumer_id']]

current_balance['balance'] = current_balance['prism_consumer_id'].map(acct_balance).fillna(0)
current_balance['update'] = current_balance['prism_consumer_id'].map(update_balance).fillna(0)
current_balance['current_balance'] = current_balance['balance'] + current_balance['update']
current_balance = current_balance.current_balance
current_balance

C:\Users\bdion\AppData\Local\Temp\ipykernel_34596\2693366838.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  current_balance['balance'] = current_balance['prism_consumer_id'].map(acct_balance).fillna(0)
C:\Users\bdion\AppData\Local\Temp\ipykernel_34596\2693366838.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  current_balance['update'] = current_balance['prism_consumer_id'].map(update_balance).fillna(0)


0         -201.22
1         5107.85
2         3235.49
3        10462.25
4        -2149.05
           ...   
13995     1873.31
13996    11173.97
13997     2444.61
13998    24322.07
13999    -1230.10
Name: current_balance, Length: 12000, dtype: float64

- Visualization for balance changes over time:

In [9]:
inflows = t_df[t_df.credit_or_debit == 'CREDIT']
outflows = t_df[t_df.credit_or_debit == 'DEBIT']

# display(inflows, outflows)
inflows.shape, outflows.shape

((878829, 6), (4258005, 6))

In [ ]:
balance_changes = t[t['is_before_balance_date'] == False].sort_values(['prism_consumer_id', 'posted_date'])
balance_changes

,prism_consumer_id,prism_transaction_id,amount,credit_or_debit,posted_date,category,balance_date,is_before_balance_date
136802,0,136738,-27.62,DEBIT,2021-03-16,FOOD_AND_BEVERAGES,NaN,False
136767,0,136703,1400.00,CREDIT,2021-03-17,TAX,NaN,False
136803,0,136739,-25.10,DEBIT,2021-03-17,FITNESS,NaN,False
136804,0,136740,-500.00,DEBIT,2021-03-17,BANKING_CATCH_ALL,NaN,False
136805,0,136741,-25.00,DEBIT,2021-03-18,FOOD_AND_BEVERAGES,NaN,False
...,...,...,...,...,...,...,...,...
6277366,13999,6275354,-2.00,DEBIT,2022-01-21,SELF_TRANSFER,NaN,False
6276795,13999,6274783,2.00,CREDIT,2022-01-24,SELF_TRANSFER,NaN,False
6277367,13999,6275355,-41.23,DEBIT,2022-01-24,FOOD_AND_BEVERAGES,NaN,False
6277368,13999,6275356,-107.98,DEBIT,2022-01-24,ENTERTAINMENT,NaN,False


## Required features:

In [ ]:
feats = t_df.groupby(['prism_consumer_id', 'category']).agg({'amount':['count', 'sum', 'std', 'mean', 'median']}).fillna(0)
feats

- Putting everything together:

In [ ]:
result = c_df[['prism_consumer_id']]
...
result